Load the objects imported from 01_load_data notebook

In [ ]:
# Define base path
base_path = "path/to/project"

In [18]:
from config import base_path, output_base

ModuleNotFoundError: No module named 'config'

In [ ]:
import spatialdata as sd
import rioxarray
import numpy as np
import matplotlib.pyplot as plt
import os

analysis_path = base_path.replace("/output/", "/analysis/")
output_base = "/path/to/plots"
experiment_id = os.path.basename(base_path)

# Auto-detect regions
regions = sorted([
    d for d in os.listdir(base_path)
    if os.path.isdir(os.path.join(base_path, d)) and d.startswith("region_")
])
print(f"Found regions: {regions}")

sdata_dict = {}
for region in regions:
    sdata_dict[region] = sd.read_zarr(f"{analysis_path}/{region}.zarr")
    print(f"{region}: loaded")

For each region, generate a low-res image with pixel grid to help identify root sections

In [ ]:
overview_dir = f"{output_base}/plots/{experiment_id}/overviews"
os.makedirs(overview_dir, exist_ok=True)

scale = 16

for region in regions:
    polyt_lowres = sdata_dict[region]["mosaic_PolyT_z3"][:, ::scale, ::scale].isel(c=0).compute()

    vmin = float(np.percentile(polyt_lowres, 1))
    vmax = float(np.percentile(polyt_lowres, 99))

    fig, ax = plt.subplots(figsize=(12, 9))
    ax.imshow(polyt_lowres, cmap="gray", vmin=vmin, vmax=vmax)
    ax.grid(True, color='red', alpha=0.3, linewidth=0.5)
    ax.set_title(f"{region} - PolyT overview")
    plt.savefig(f"{overview_dir}/{region}_polyt_overview.png", dpi=200)
    plt.close()
    print(f"{region}: overview saved")

Check one gene per moisaic to see which sections have good expression to subsequently crop individually

In [ ]:
verify_gene = "Solyc04g081490"  # change to any gene you expect to be expressed

for region in regions:
    points_layer = [k for k in sdata_dict[region].points.keys()][0]
    df_r = sdata_dict[region][points_layer].compute()
    
    polyt_lowres = sdata_dict[region]["mosaic_PolyT_z3"][:, ::scale, ::scale].isel(c=0).compute()
    vmin = float(np.percentile(polyt_lowres, 1))
    vmax = float(np.percentile(polyt_lowres, 99))

    gene_subset = df_r[df_r["gene"] == verify_gene]

    fig, ax = plt.subplots(figsize=(12, 9))
    ax.imshow(polyt_lowres, cmap="gray", vmin=vmin, vmax=vmax)
    
    # Overlay transcripts scaled to low-res coordinates
    ax.scatter(
        gene_subset["x"] / scale,
        gene_subset["y"] / scale,
        s=0.5, c="crimson", alpha=0.7, rasterized=True
    )
    ax.set_title(f"{region} - {verify_gene} (n={len(gene_subset)})")
    ax.axis("off")
    plt.savefig(f"{overview_dir}/{region}_{verify_gene}_overview.png", dpi=200)
    plt.close()
    print(f"{region}: done")

Coordinates of sections within the overview images

In [ ]:
roots = {
    "region_R3": {
        "root1": (670, 1500, 1400, 1700),
        "root2": (2850, 3030, 2225, 2375),
    },
    "region_R4": {
        "root1": (1050, 1250, 1820, 2000),
        "root2": (1350, 1500, 3425, 3600),
        "root3": (1400, 1550, 4875, 5025),
    },
    "region_R5": {
        "root1": (970, 1120, 3790, 3950),
        "root2": (1200, 1320, 4000, 4120),
        "root3": (1620, 1800, 3000, 3200),
        "root4": (920, 1120, 1320, 1520),
        "root5": (1250, 1450, 1120, 1320),
    },
    "region_R6": {
        "root1": (1590, 1710, 640, 760),
        "root2": (2850, 3600, 1075, 1320),
    },
}

Check one gene per root per region to see if crop is okay

In [ ]:
import harpy as hp
import spatialdata as sd
import matplotlib.pyplot as plt
from matplotlib.colors import Normalize
import numpy as np
import os

verify_gene = "Solyc04g081490"
verify_dir = f"{output_base}/plots/verify"
os.makedirs(verify_dir, exist_ok=True)

for region, region_roots in roots.items():
    print(f"Loading {region}...")
    sdata_r = sd.read_zarr(f"{output_base}/sdata_{region}.zarr")
    
    img_layer = [k for k in sdata_r.images.keys()][0]
    points_layer = [k for k in sdata_r.points.keys()][0]
    df_r = sdata_r[points_layer].compute()

    for root_name, (xmin, xmax, ymin, ymax) in region_roots.items():
        xmin_s, xmax_s, ymin_s, ymax_s = xmin*scale, xmax*scale, ymin*scale, ymax*scale

        crop = sdata_r[img_layer][:, ymin_s:ymax_s, xmin_s:xmax_s].isel(c=0).compute()
        norm = Normalize(vmin=float(np.percentile(crop, 1)), vmax=float(np.percentile(crop, 99)))
        del crop

        subset = df_r[(df_r["gene"] == verify_gene) &
                      (df_r["x"] > xmin_s) & (df_r["x"] < xmax_s) &
                      (df_r["y"] > ymin_s) & (df_r["y"] < ymax_s)]

        fig, ax = plt.subplots(figsize=(8, 8))
        hp.pl.plot_sdata(
            sdata_r,
            img_layer=img_layer,
            channel=0,
            crd=[xmin_s, xmax_s, ymin_s, ymax_s],
            to_coordinate_system="global",
            ax=ax,
            render_images_kwargs={"cmap": "gray", "norm": norm},
        )
        ax.scatter(subset["x"], subset["y"], s=1, c="crimson", alpha=0.8)
        ax.set_title(f"{region} - {root_name} - {verify_gene} (n={len(subset)})")

        plt.savefig(f"{verify_dir}/{region}_{root_name}_{verify_gene}.png", dpi=150, bbox_inches="tight")
        plt.close()
        print(f"  {root_name}: done")

    del sdata_r, df_r
    print(f"{region}: complete")

Run all genes for all roots of all regions for JA - group them per gene. Individual plots

In [ ]:
import harpy as hp
import spatialdata as sd
import matplotlib.pyplot as plt
from matplotlib.colors import Normalize
import numpy as np
import os
import gc

scale = 16
output_base = "/home/ninte/projects/spatial_analysis"
for region, region_roots in roots.items():
    print(f"Loading {region}...")
    sdata_r = sd.read_zarr(f"{output_base}/sdata_{region}.zarr")
    
    img_layer = [k for k in sdata_r.images.keys()][0]
    points_layer = [k for k in sdata_r.points.keys()][0]
    df_r = sdata_r[points_layer].compute()
    panel_genes_r = [g for g in df_r["gene"].unique() if "blank" not in g.lower()]

    for root_name, (xmin, xmax, ymin, ymax) in region_roots.items():
        xmin_s, xmax_s, ymin_s, ymax_s = xmin*scale, xmax*scale, ymin*scale, ymax*scale

        crop = sdata_r[img_layer][:, ymin_s:ymax_s, xmin_s:xmax_s].isel(c=0).compute()
        norm = Normalize(vmin=float(np.percentile(crop, 1)), vmax=float(np.percentile(crop, 99)))
        del crop

        for gene in panel_genes_r:
            plot_dir = f"{output_base}/plots/genes/{gene}"
            os.makedirs(plot_dir, exist_ok=True)
            
            out_path = f"{plot_dir}/{gene}_{region}_{root_name}.png"
            if os.path.exists(out_path):
                continue

            subset = df_r[(df_r["gene"] == gene) &
                          (df_r["x"] > xmin_s) & (df_r["x"] < xmax_s) &
                          (df_r["y"] > ymin_s) & (df_r["y"] < ymax_s)]

            fig, ax = plt.subplots(figsize=(8, 8))
            hp.pl.plot_sdata(
                sdata_r,
                img_layer=img_layer,
                channel=0,
                crd=[xmin_s, xmax_s, ymin_s, ymax_s],
                to_coordinate_system="global",
                ax=ax,
                render_images_kwargs={"cmap": "gray", "norm": norm},
            )
            ax.scatter(subset["x"], subset["y"], s=1, c="crimson", alpha=0.8)
            ax.set_title(f"{gene} | {region} | {root_name} (n={len(subset)})")

            plt.savefig(out_path, dpi=150, bbox_inches="tight")
            plt.close('all')

    del sdata_r, df_r
    gc.collect()
    print(f"{region}: complete")

Plot all regions per gene together, optional

In [ ]:
import math

# Collect all region/root combinations
all_combos = [(region, root_name, coords) 
              for region, region_roots in roots.items() 
              for root_name, coords in region_roots.items()]

n_panels = len(all_combos)
n_cols = 3  # adjust to your preference
n_rows = math.ceil(n_panels / n_cols)

for gene in panel_genes_r:  # or use a master gene list
    out_path = f"{output_base}/plots/genes/{gene}/{gene}_all_regions.png"
    #if os.path.exists(out_path):
    #    continue
    
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(6*n_cols, 6*n_rows))
    axes = axes.flatten()
    
    # hide any unused panels
    for i in range(n_panels, len(axes)):
        axes[i].axis("off")

    for ax, (region, root_name, (xmin, xmax, ymin, ymax)) in zip(axes, all_combos):
        xmin_s, xmax_s, ymin_s, ymax_s = xmin*scale, xmax*scale, ymin*scale, ymax*scale

        sdata_r = sd.read_zarr(f"{output_base}/sdata_{region}.zarr")
        img_layer = [k for k in sdata_r.images.keys()][0]
        points_layer = [k for k in sdata_r.points.keys()][0]
        df_r = sdata_r[points_layer].compute()

        crop = sdata_r[img_layer][:, ymin_s:ymax_s, xmin_s:xmax_s].isel(c=0).compute()
        norm = Normalize(vmin=float(np.percentile(crop, 1)), vmax=float(np.percentile(crop, 99)))
        del crop

        subset = df_r[(df_r["gene"] == gene) &
                      (df_r["x"] > xmin_s) & (df_r["x"] < xmax_s) &
                      (df_r["y"] > ymin_s) & (df_r["y"] < ymax_s)]

        hp.pl.plot_sdata(
            sdata_r,
            img_layer=img_layer,
            channel=0,
            crd=[xmin_s, xmax_s, ymin_s, ymax_s],
            to_coordinate_system="global",
            ax=ax,
            render_images_kwargs={"cmap": "gray", "norm": norm},
        )
        ax.scatter(subset["x"], subset["y"], s=1, c="crimson", alpha=0.8)
        ax.set_title(f"{region}\n{root_name} (n={len(subset)})", fontsize=9)
        ax.axis("off")

        del sdata_r, df_r
        gc.collect()

    fig.suptitle(gene, fontsize=12, y=1.02)
    plt.tight_layout()
    plt.savefig(out_path, dpi=150, bbox_inches="tight")
    plt.close('all')
    print(f"{gene}: combined plot saved")

For sequentially-imaged genes, load the mosaics and visualize per root per region

In [ ]:
import harpy as hp
import spatialdata as sd
import matplotlib.pyplot as plt
from matplotlib.colors import Normalize
import numpy as np
import os
import gc

scale = 16
seq_genes = ["Solyc02g085730", "Solyc04g082030"]

for gene in seq_genes:
    plot_dir = f"{output_base}/plots/genes/{gene}"
    os.makedirs(plot_dir, exist_ok=True)

    for region, region_roots in roots.items():
        sdata_r = sd.read_zarr(f"{output_base}/sdata_{region}.zarr")
        img_layer = [k for k in sdata_r.images.keys()][0]
        image_dir = f"{base_path}/{region}/images"

        # Load gene channel lazily
        gene_tif = rioxarray.open_rasterio(
            f"{image_dir}/mosaic_{gene}_z3.tif",
            chunks=(1, 4096, 4096), lock=False
        ).rename({"band": "c"})

        for root_name, (xmin, xmax, ymin, ymax) in region_roots.items():
            xmin_s, xmax_s, ymin_s, ymax_s = xmin*scale, xmax*scale, ymin*scale, ymax*scale

            out_path = f"{plot_dir}/{gene}_{region}_{root_name}.png"
            if os.path.exists(out_path):
                continue

            # PolyT crop
            polyt_crop = sdata_r[img_layer][:, ymin_s:ymax_s, xmin_s:xmax_s].isel(c=0).compute()
            polyt_norm = Normalize(vmin=float(np.percentile(polyt_crop, 1)), vmax=float(np.percentile(polyt_crop, 99)))
            del polyt_crop

            # Gene channel crop
            gene_crop = gene_tif.isel(c=0)[ymin_s:ymax_s, xmin_s:xmax_s].compute()
            gene_norm = Normalize(vmin=float(np.percentile(gene_crop, 1)), vmax=float(np.percentile(gene_crop, 99)))
            del gene_crop

            fig, axes = plt.subplots(1, 2, figsize=(12, 6))

            # PolyT crop for plotting
            polyt_plot = sdata_r[img_layer][:, ymin_s:ymax_s, xmin_s:xmax_s].isel(c=0).compute()
            polyt_norm = Normalize(vmin=float(np.percentile(polyt_plot, 1)), vmax=float(np.percentile(polyt_plot, 99)))

            # Gene channel crop
            gene_plot = gene_tif.isel(c=0)[ymin_s:ymax_s, xmin_s:xmax_s].compute()
            gene_norm = Normalize(vmin=float(np.percentile(gene_plot, 1)), vmax=float(np.percentile(gene_plot, 99)))

            fig, axes = plt.subplots(1, 2, figsize=(12, 6))

            axes[0].imshow(polyt_plot, cmap="gray", norm=polyt_norm, origin="upper")
            axes[0].set_title("PolyT")
            axes[0].axis("off")

            axes[1].imshow(gene_plot, cmap="hot", norm=gene_norm, origin="upper")
            axes[1].set_title(gene)
            axes[1].axis("off")

            fig.suptitle(f"{gene} | {region} | {root_name}", fontsize=12)
            plt.tight_layout()
            plt.savefig(out_path, dpi=150, bbox_inches="tight")
            plt.close('all')

        del sdata_r, gene_tif
        gc.collect()
        print(f"{gene} - {region}: done")